# 07 — MuJoCo Menagerie Flexiv Rizon4 Data + Diffusion Training

This notebook loads Flexiv Rizon4 from MuJoCo Menagerie assets, generates trajectory data, trains a diffusion model, and samples generated trajectories.


### Setup

Install dependencies, set seeds, and define paths for assets/data/config/output.


In [ ]:
from pathlib import Path
import os
import random
import subprocess
import sys

import numpy as np
import torch

# If the repo is not present in /content, set REPO_URL to your GitHub repo and rerun.
REPO_URL = "https://github.com/<your-org>/score-manifold-optimization.git"
REPO_DIR = Path("/content/score-manifold-optimization")

if not (REPO_DIR / "pyproject.toml").exists():
    if "<" in REPO_URL:
        raise RuntimeError(
            "Repo not found at /content/score-manifold-optimization. "
            "Please set REPO_URL to your repository URL or clone manually."
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[mujoco]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "imageio[ffmpeg]"], check=True)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = REPO_DIR / "data"
OUTPUT_DIR = REPO_DIR / "outputs/rizon4_train_demo"
CONFIG_PATH = REPO_DIR / "configs/train_rizon4_quick.yaml"
XML_PATH = REPO_DIR / "examples/assets/flexiv_rizon4/scene.xml"
DATASET_PATH = DATA_DIR / "rizon4_T100_n20000.pt"

print(f"Repo: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"XML path: {XML_PATH}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Config path: {CONFIG_PATH}")
print(f"Output dir: {OUTPUT_DIR}")


### Local Setup (Alternative)

Use this when running locally with the repository already available.


In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import torch

start = Path.cwd().resolve()
REPO_DIR = next((p for p in [start, *start.parents] if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    raise RuntimeError("Could not find repo root (missing pyproject.toml in parent dirs).")

os.chdir(REPO_DIR)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = REPO_DIR / "data"
OUTPUT_DIR = REPO_DIR / "outputs/rizon4_train_demo"
CONFIG_PATH = REPO_DIR / "configs/train_rizon4_quick.yaml"
XML_PATH = REPO_DIR / "examples/assets/flexiv_rizon4/scene.xml"
DATASET_PATH = DATA_DIR / "rizon4_T100_n20000.pt"

print(f"Repo root: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"XML path: {XML_PATH}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Config path: {CONFIG_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

# Install once from terminal (repo root):
# pip install -e .[mujoco]
# pip install imageio[ffmpeg]


### Load MuJoCo Flexiv Rizon4

Instantiate the MuJoCo-backed control system and inspect dimensions.


In [ ]:
from diffusion.control.mujoco import MuJoCoSystem

D_T = 0.005
T_S = 0.05

system = MuJoCoSystem(model_path=str(XML_PATH), dT=D_T, Ts=T_S)

print(f"state_dim: {system.get_state_dim()}")
print(f"input_dim: {system.get_input_dim()}")
print(f"output_dim: {system.get_output_dim()}")
print(f"dT: {system.dT}, Ts: {system.Ts}, frame_skip: {system.frame_skip}")


### Generate Dataset

Sample controls within actuator limits, simulate trajectories, and save a canonical dataset dict.


In [ ]:
import matplotlib.pyplot as plt

DATA_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 100
N_TRAIN = 20000
N_TEST = 2000
N_VAL = 1000
N_TOTAL = N_TRAIN + N_TEST + N_VAL

ctrlrange_np = system.model.actuator_ctrlrange
ctrl_low = torch.tensor(ctrlrange_np[:, 0], dtype=torch.float32)
ctrl_high = torch.tensor(ctrlrange_np[:, 1], dtype=torch.float32)

u_rand = torch.rand(N_TOTAL, HORIZON, system.get_input_dim(), dtype=torch.float32)
u = ctrl_low.view(1, 1, -1) + (ctrl_high - ctrl_low).view(1, 1, -1) * u_rand

# Start every rollout from MuJoCo model default state [qpos0, qvel0].
x0_single = torch.cat(
    [
        torch.tensor(system.model.qpos0, dtype=torch.float32),
        torch.zeros(system.model.nv, dtype=torch.float32),
    ],
    dim=0,
)
x0 = x0_single.unsqueeze(0).repeat(N_TOTAL, 1)

with torch.no_grad():
    y = system.simulate_out(x0, u)

trajectories = torch.cat([u, y], dim=-1)

train_data = trajectories[:N_TRAIN]
test_data = trajectories[N_TRAIN:N_TRAIN + N_TEST]
val_data = trajectories[N_TRAIN + N_TEST:]

metadata = {
    "space": {
        "class": "TrajectorySpace",
        "params": {
            "horizon": HORIZON,
            "input_dim": int(system.get_input_dim()),
            "output_dim": int(system.get_output_dim()),
        },
    },
    "constraint": {
        "class": "MuJoCoSystem",
        "params": {
            "model_path": str(XML_PATH),
            "dT": float(system.dT),
            "Ts": float(system.Ts),
        },
    },
    "state_dim": int(system.get_state_dim()),
    "n_train": int(N_TRAIN),
    "n_test": int(N_TEST),
    "n_val": int(N_VAL),
    "created": __import__("datetime").datetime.now().isoformat(),
    "version": "1.0",
    "generation_config": {
        "horizon": HORIZON,
        "control_range": ctrlrange_np.tolist(),
        "x0": "model_default_state",
        "dT": float(system.dT),
        "Ts": float(system.Ts),
    },
}

dataset = {
    "train_data": train_data,
    "test_data": test_data,
    "val_data": val_data,
    "metadata": metadata,
}
torch.save(dataset, DATASET_PATH)

print(f"Saved dataset: {DATASET_PATH}")
print(f"train shape: {tuple(train_data.shape)}")
print(f"test shape:  {tuple(test_data.shape)}")
print(f"val shape:   {tuple(val_data.shape)}")

# Plot one trajectory (first 2 state channels and first 2 controls)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_data[0, :, system.get_input_dim():system.get_input_dim() + 2])
axes[0].set_title("State channels 0-1")
axes[0].grid(alpha=0.3)

axes[1].plot(train_data[0, :, :2])
axes[1].set_title("Control channels 0-1")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Visualize One Training Trajectory (MP4)

Render one training trajectory with MuJoCo and preview it inline.


In [ ]:
import imageio.v2 as imageio
import mujoco
from IPython.display import Video
from diffusion.data import load_dataset

dataset = load_dataset(str(DATASET_PATH), device="cpu")
train_data = dataset.train_data
traj_idx = 128
y_demo = train_data[traj_idx, :, system.get_input_dim():].detach().cpu()

model = mujoco.MjModel.from_xml_path(str(XML_PATH))
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model)

frames = []
for t in range(y_demo.shape[0]):
    state_t = y_demo[t]
    data.qpos[:] = state_t[:model.nq].numpy()
    data.qvel[:] = state_t[model.nq:model.nq + model.nv].numpy()
    mujoco.mj_forward(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

renderer.close()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
mp4_path = OUTPUT_DIR / f"rizon4_demo_traj{traj_idx}.mp4"
fps = int(round(1.0 / system.Ts))
imageio.mimsave(mp4_path, frames, fps=fps)

print(f"Saved MP4: {mp4_path}")
Video(str(mp4_path), embed=True)


### Train Model

Train the diffusion model on generated trajectories (this is a long-running cell).


In [ ]:
import json
from datetime import datetime
import yaml

from diffusion.data import load_dataset
from diffusion.models import create_model
from diffusion.training import DiffusionTrainer, TrainingOptions, create_diffusion, create_optimizer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset = load_dataset(str(DATASET_PATH), device="cpu")
space = dataset.get_space()
diffusion = create_diffusion(space, cfg)
model = create_model(
    model_type=str(cfg["model"]["type"]),
    space=space,
    diffusion=diffusion,
    config=cfg,
    initialize=True,
).to(DEVICE)
optimizer = create_optimizer(model, cfg)

options_dict = dict(cfg.get("training_options", {}))
options_dict["batch_size"] = int(cfg.get("training", {}).get("batch_size", 32))
options = TrainingOptions(**options_dict)

constraint = dataset.constraint
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

def log_fn(samples):
    # Warning: MuJoCo rollout for violation is CPU/NumPy-based, so this log step moves data across CPU/GPU.
    # If you want to avoid this, set e.g. log_fn = lambda samples: {}
    violations = constraint.violation(samples)
    return {
        "constraint_violation_mean": violations.mean().item(),
        "constraint_violation_std": violations.std().item(),
        "constraint_violation_max": violations.max().item(),
    }

trainer = DiffusionTrainer(
    diffusion=diffusion,
    score_model=model,
    train_data=dataset.train_data,
    optimizer=optimizer,
    options=options,
    log_fn=log_fn,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    metadata=dataset.metadata,
)

num_epochs = int(cfg.get("training", {}).get("num_epochs", 50000))
trainer.train(num_epochs=num_epochs)
trainer.save_checkpoint(str(OUTPUT_DIR / "checkpoint.pt"))

with open(OUTPUT_DIR / "config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

summary = {
    "timestamp": datetime.now().isoformat(),
    "output_dir": str(OUTPUT_DIR),
    "num_epochs": num_epochs,
    "final_loss": float(trainer.train_losses[-1]) if trainer.train_losses else None,
    "num_params": num_params,
    "dataset_path": str(DATASET_PATH),
    "model_type": str(cfg["model"]["type"]),
}
with open(OUTPUT_DIR / "train_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Training complete. Artifacts written to {OUTPUT_DIR}")


### Verify Outputs

Check that expected checkpoint artifacts were written.


In [ ]:
import json

required = ["model.pth", "checkpoint_data.pt", "config.yaml", "metadata.json", "train_summary.json"]
missing = [name for name in required if not (OUTPUT_DIR / name).exists()]

if missing:
    print("Missing artifacts:")
    for name in missing:
        print(f"  - {name}")
else:
    print("All expected artifacts are present.")

with open(OUTPUT_DIR / "train_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))


### Sample + Dynamics Error

Sample trajectories from the trained model and compare generated outputs to true MuJoCo rollouts.


In [ ]:
N_SAMPLES = 4
import yaml
from diffusion.data import load_dataset
from diffusion.models import create_model
from diffusion.training import create_diffusion

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset = load_dataset(str(DATASET_PATH), device="cpu")
space = dataset.get_space()
diffusion = create_diffusion(space, cfg)
constraint = system.to_constraint(space.horizon)
model = create_model(
    model_type=str(cfg["model"]["type"]),
    space=space,
    diffusion=diffusion,
    config=cfg,
    initialize=False,
).to(DEVICE)

state_dict = torch.load(OUTPUT_DIR / "model.pth", map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

with torch.no_grad():
    samples = diffusion.sample_reverse(
        N_sample=N_SAMPLES,
        score_model=model,
        flow_type="ODE",
        end_only=True,
        device=DEVICE,
    )

u, y_gen = constraint.split_input_output(samples)
x0_default = torch.cat(
    [
        torch.tensor(system.model.qpos0, device=samples.device, dtype=samples.dtype),
        torch.zeros(system.model.nv, device=samples.device, dtype=samples.dtype),
    ],
    dim=0,
).unsqueeze(0).repeat(N_SAMPLES, 1)


with torch.no_grad():
    y_true = system.simulate_out(x0_default, u)

err = y_gen - y_true
mae = err.abs().mean(dim=(1, 2)).detach().cpu()
rmse = err.pow(2).mean(dim=(1, 2)).sqrt().detach().cpu()
violation = constraint.violation(samples).detach().cpu()

print(f"Samples shape: {tuple(samples.shape)}")
print(
    "Output error (generated vs true) | "
    f"MAE mean={mae.mean().item():.4e}, median={mae.median().item():.4e}, max={mae.max().item():.4e}"
)
print(
    "Output error (generated vs true) | "
    f"RMSE mean={rmse.mean().item():.4e}, median={rmse.median().item():.4e}, max={rmse.max().item():.4e}"
)
print(
    "Dynamics violation | "
    f"mean={violation.mean().item():.4e}, median={violation.median().item():.4e}, max={violation.max().item():.4e}"
)


### Plot Example Trajectories

Plot a few generated vs true joint trajectories over time.


In [ ]:
import matplotlib.pyplot as plt

num_plot = min(3, N_SAMPLES)
fig, axes = plt.subplots(2, num_plot, figsize=(5 * num_plot, 6), squeeze=False)

for i in range(num_plot):
    ax1 = axes[0, i]
    ax2 = axes[1, i]

    ax1.plot(y_true[i, :, 0].detach().cpu(), "k--", label="Simulated")
    ax1.plot(y_gen[i, :, 0].detach().cpu(), color="tab:blue", label="Generated")
    ax1.set_title(f"Trajectory {i} | joint 1")
    ax1.set_xlabel("t")
    ax1.set_ylabel("q1")
    ax1.grid(alpha=0.3)
    ax1.legend()

    ax2.plot(y_true[i, :, 1].detach().cpu(), "k--", label="Simulated")
    ax2.plot(y_gen[i, :, 1].detach().cpu(), color="tab:orange", label="Generated")
    ax2.set_title(f"Trajectory {i} | joint 2")
    ax2.set_xlabel("t")
    ax2.set_ylabel("q2")
    ax2.grid(alpha=0.3)
    ax2.legend()

plt.tight_layout()
plt.show()
